# HotpotQA inference from a saved AutoSchemaKG ZIP

Upload the ZIP produced by the HotpotQA construction notebook, inspect its GraphML graph, run KG-only QA with local `Qwen/Qwen3.5-2B`, calculate EM/F1, and download a new archive. This notebook does not rebuild the graph.

## Check GPU before uploading
Changing runtime type later resets `/content`, so select a GPU now.

In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode != 0:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before continuing')

## Upload the construction-result ZIP
Upload only the AutoSchemaKG result archive, not a ZIP of the entire `/content` directory.

In [ ]:
from pathlib import Path
from google.colab import files
uploaded = files.upload()
zip_paths = [Path('/content', name) for name in uploaded if name.lower().endswith('.zip')]
if len(zip_paths) != 1:
    raise RuntimeError(f'Upload exactly one ZIP; received {zip_paths}')
ZIP_PATH = zip_paths[0]
print('Input archive:', ZIP_PATH)

## Clone and install

In [ ]:
import os
REPO_DIR = '/content/SmallScaledAutoSchemaKG'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', 'https://github.com/phuongth05/SmallScaledAutoSchemaKG.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', 'pull', '--ff-only'], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', 'requirements-colab.txt'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '--pre', 'vllm', '--torch-backend=auto'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchaudio', 'torchvision'], check=False)
subprocess.run(['uv', 'pip', 'install', '--system', 'torchvision', '--torch-backend=auto'], check=True)
subprocess.run([sys.executable, '-c', "import torch, torchvision, vllm; print('torch', torch.__version__, 'CUDA', torch.version.cuda, 'torchvision', torchvision.__version__, 'vLLM', vllm.__version__)"], check=True)

## Start local Qwen server

In [ ]:
import shutil, time
import requests
MODEL_ID = 'Qwen/Qwen3.5-2B'
PORT = 8000
LOG_PATH = '/content/qwen35_qa_vllm.log'
vllm_executable = shutil.which('vllm')
if not vllm_executable:
    raise RuntimeError('vLLM was not installed')
def server_is_ready():
    try:
        return requests.get(f'http://127.0.0.1:{PORT}/v1/models', timeout=5).ok
    except requests.RequestException:
        return False
if not server_is_ready():
    server_log = open(LOG_PATH, 'w', encoding='utf-8')
    command = [vllm_executable, 'serve', MODEL_ID, '--host', '127.0.0.1', '--port', str(PORT), '--dtype', 'half', '--max-model-len', '4096', '--gpu-memory-utilization', '0.80', '--language-model-only']
    server = subprocess.Popen(command, stdout=server_log, stderr=subprocess.STDOUT)
    deadline = time.time() + 900
    while time.time() < deadline:
        if server.poll() is not None:
            server_log.flush()
            print(Path(LOG_PATH).read_text(encoding='utf-8', errors='replace')[-10000:])
            raise RuntimeError(f'vLLM stopped with code {server.returncode}')
        if server_is_ready():
            print('Local Qwen server is ready')
            break
        time.sleep(5)
    else:
        raise TimeoutError(f'vLLM did not start; inspect {LOG_PATH}')
else:
    print('Existing local server is ready')

## Restore, visualize, retrieve graph triples, and answer

In [ ]:
WORK_DIR = '/content/hotpotqa_from_zip'
OUTPUT_ZIP = '/content/autoschemakg_hotpotqa_reanalyzed.zip'
subprocess.run([
    sys.executable, '-u', 'scripts/run_hotpotqa_from_zip.py', str(ZIP_PATH),
    '--work-dir', WORK_DIR, '--output-zip', OUTPUT_ZIP,
    '--model', MODEL_ID, '--base-url', f'http://127.0.0.1:{PORT}/v1',
    '--top-k', '60', '--overwrite'
], check=True)

In [ ]:
import json
from IPython.display import Image, display
result_path = Path(WORK_DIR, 'hotpotqa_kg_qa_results.json')
evaluation = json.loads(result_path.read_text(encoding='utf-8'))
print(json.dumps({key: value for key, value in evaluation.items() if key != 'results'}, indent=2))
for result in evaluation['results']:
    assert result['retrieved_triple_count'] > 0
    print('\nQuestion:', result['question'])
    print('Gold:', result['gold_answer'])
    print('Prediction:', result['prediction'])
    print('Retrieved triples:', result['retrieved_triple_count'])
display(Image(filename=str(Path(WORK_DIR, 'hotpotqa_graph_overview.png'))))

## Download the reanalyzed archive
The archive contains the original graph/provenance, a graph PNG, and corrected QA results.

In [ ]:
assert Path(OUTPUT_ZIP).is_file()
files.download(OUTPUT_ZIP)